

**The Hybrid Equation: KNOWN PHYSICS + UDE**
$$ C_m \frac{dV}{dt} = I_{ext} - \mathbf{NN(V)} - I_{K}(V) - I_{L}(V) $$


In [1]:
using SciMLSensitivity, DifferentialEquations, Zygote, Optimisers, Flux
using Optimization, OptimizationOptimisers, LinearAlgebra, DataFrames, JLD2, Random, Statistics, Printf

Random.seed!(1234)

# Load data and cast to Float32 immediately
@load "Data/synthetic_data/noise_0_hh_2d_model.jld" V 
V = Float32.(V) 

println("Environment ready. Data (Float32) points: ", length(V))

Environment ready. Data (Float32) points: 501


In [2]:
const Cm = 1.0f0
const g_Na = 120.0f0
const E_Na = 50.0f0
const g_L = 0.3f0
const E_L = -54.387f0
const g_K = 36.0f0
const E_K = -77.0f0

# Rate functions using Float32 literals
alpha_n(V) = 0.01f0 * (V + 55.0f0) / (1.0f0 - exp(-(V + 55.0f0) / 10.0f0))
beta_n(V)  = 0.125f0 * exp(-(V + 65.0f0) / 80.0f0)
alpha_m(V) = 0.1f0 * (V + 40.0f0) / (1.0f0 - exp(-(V + 40.0f0) / 10.0f0))
beta_m(V)  = 4.0f0 * exp(-(V + 65.0f0) / 18.0f0)
alpha_h(V) = 0.07f0 * exp(-(V + 65.0f0) / 20.0f0)
beta_h(V)  = 1.0f0 / (1.0f0 + exp(-(V + 35.0f0) / 10.0f0))

# Dynamics helpers
n_inf(V) = alpha_n(V) / (alpha_n(V) + beta_n(V))
tau_n(V) = 1.0f0 / (alpha_n(V) + beta_n(V))

tau_n (generic function with 1 method)

In [8]:
# Neural Network for unknown Sodium current
NN_Model = Chain(
    Dense(1, 32, tanh), 
    Dense(32, 16,tanh),
    Dense(16, 1)
) |> f32 

p_nn, re = Flux.destructure(NN_Model)

# Normalization for numerical stability in Float32
const V_mean = mean(V)
const V_std = std(V)
norm_input(v) = (v - V_mean) / V_std

function Stimulus(t)
    # Match the pulses in the synthetic data
    if (t >= 10.0f0 && t < 11.0f0) || (t >= 30.0f0 && t < 31.0f0)
        return 20.0f0
    else
        return 0.0f0
    end
end

Stimulus (generic function with 1 method)

In [9]:
function hodgkin_huxley_UDE!(du, u, p, t)
    V_curr, n = u
    nn_model = re(p)
    
    # NN Input normalization is critical for Float32 gradients
    pred_I_Na = nn_model([norm_input(V_curr)])[1]
    
    I_ext = Stimulus(t)
    I_K = g_K * n^4 * (V_curr - E_K)
    I_L = g_L * (V_curr - E_L)
    
    du[1] = (I_ext - pred_I_Na - I_K - I_L) / Cm
    du[2] = (n_inf(V_curr) - n) / tau_n(V_curr)
end

u0_true = Float32[-65.0, n_inf(-65.0f0)]
t_span = (0.0f0, 50.0f0)
t_train = range(t_span[1], t_span[2], length=length(V))

prob_nn = ODEProblem(hodgkin_huxley_UDE!, u0_true, t_span, p_nn)

ODEProblem with uType Vector{Float32} and tType Float32. In-place: true
Non-trivial mass matrix: false
timespan: (0.0f0, 50.0f0)
u0: 2-element Vector{Float32}:
 -65.0
   0.3176769

In [10]:
function predict_ude(p)
    # remake() is used to update parameters for each training step
    _prob = remake(prob_nn, p=p)
    
    # Use TRBDF2() or Rodas5P() with relaxed Float32 tolerances
    solve(_prob, TRBDF2(), saveat=t_train, 
          reltol=1f-3, abstol=1f-4, 
          dtmin=1f-7, dtmax=1f-1,
          sensealg=InterpolatingAdjoint(autojacvec=ZygoteVJP()))
end

function loss(p)
    pred = predict_ude(p)
    if pred.retcode != :Success
        return 1f6 # Penalty for unstable solutions [cite: 153]
    end
    
    # Calculate MSE on the Voltage trace
    return sum(abs2, pred[1, :] .- V) / length(V) 
end

loss (generic function with 1 method)

In [11]:
losses = Float32[]
min_loss = Inf
const SAVE_PATH = "ude_best_model_f32.jld2"

function callback(state, l)
    push!(losses, l)
    iter = length(losses)
    
    # Check if we have a new best model
    is_new_best = l < min_loss
    if is_new_best
        global min_loss = l
        jldsave(SAVE_PATH; params=state.u, loss_history=losses, best_loss=l)
    end
    
    if iter % 10 == 0 || is_new_best
        @printf("Iter: %4d | Loss: %.5e %s\n", iter, l, is_new_best ? "NEW BEST!" : "")
    end
    return false
end

# Optimization Setup with Zygote Automatic Differentiation
optf = Optimization.OptimizationFunction((x, p) -> loss(x), Optimization.AutoZygote())
optprob = Optimization.OptimizationProblem(optf, p_nn)

# Train for 1000 iterations [cite: 213]
res_adam = Optimization.solve(
    optprob, 
    OptimizationOptimisers.Adam(0.01), 
    callback=callback, 
    maxiters=500
)

Iter:    1 | Loss: 5.22875e+01 NEW BEST!
Iter:    2 | Loss: 5.09444e+01 NEW BEST!
Iter:    3 | Loss: 4.95642e+01 NEW BEST!
Iter:    4 | Loss: 4.81555e+01 NEW BEST!
Iter:    5 | Loss: 4.67375e+01 NEW BEST!
Iter:    6 | Loss: 4.53299e+01 NEW BEST!
Iter:    7 | Loss: 4.39583e+01 NEW BEST!
Iter:    8 | Loss: 4.26586e+01 NEW BEST!
Iter:    9 | Loss: 4.14739e+01 NEW BEST!
Iter:   10 | Loss: 4.04497e+01 NEW BEST!
Iter:   11 | Loss: 3.96404e+01 NEW BEST!
Iter:   12 | Loss: 3.91284e+01 NEW BEST!
Iter:   13 | Loss: 3.90198e+01 NEW BEST!
Iter:   19 | Loss: 3.87997e+01 NEW BEST!
Iter:   20 | Loss: 3.82828e+01 NEW BEST!
Iter:   21 | Loss: 3.78347e+01 NEW BEST!
Iter:   22 | Loss: 3.75164e+01 NEW BEST!
Iter:   23 | Loss: 3.73296e+01 NEW BEST!
Iter:   24 | Loss: 3.72152e+01 NEW BEST!
Iter:   25 | Loss: 3.71188e+01 NEW BEST!
Iter:   26 | Loss: 3.70057e+01 NEW BEST!
Iter:   27 | Loss: 3.68566e+01 NEW BEST!
Iter:   28 | Loss: 3.66622e+01 NEW BEST!
Iter:   29 | Loss: 3.64208e+01 NEW BEST!
Iter:   30 | Los

retcode: Default
u: 609-element Vector{Float32}:
  0.2680868
 -0.27860945
 -0.151755
 -0.4490589
  0.30269462
 -0.50967413
  0.40908757
 -0.3596755
  0.42124653
 -0.2372948
  ⋮
  3.021959
 -2.837356
 -2.9705467
  3.130917
 -3.1497643
 -2.9380631
  3.1214435
 -2.4840174
 -2.597609